# 04 - Joint Fine-Tuning (DF + Phi)

Joint fine-tuning of the distribution function and gravitational potential:

$$L = \lambda_{\mathrm{cbe}}\,L_{\mathrm{CBE}} + \lambda_{\mathrm{nll}}\,\mathrm{NLL}$$

**Prerequisites**: Completed DF run (`02_train_df`) and Phi run (`03_train_phi`).

In [ ]:
import jax
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
from dpjax.config import load_config, merge_config
from dpjax.paths import PROJECT_ROOT, DATA_DIR, RUNS_DIR

print(f"Project root: {PROJECT_ROOT}")

## 1. Configure Joint Training

In [ ]:
cfg = load_config("configs/joint_plummer.yaml")

# Override for a quick test
cfg = merge_config(cfg, {
    "train": {
        "epochs": 2,
        "batch_size": 2048,
        "log_every": 20,
        "ckpt_every": 100,
        "mode": "alt",        # "alt" or "both"
        "lambda_cbe": 1.0,
        "lambda_nll": 0.3,
        "lr_df": 1e-4,
        "lr_phi": 1e-4,
    }
})

import yaml
print(yaml.safe_dump(cfg, sort_keys=False))

## 2. Run Joint Fine-Tuning

In [ ]:
from experiments.finetune_joint import run_joint_finetuning

DATA_PATH    = DATA_DIR / "plummer_n131072.h5"
DF_RUN_DIR   = RUNS_DIR / "plummer" / "df"
PHI_RUN_DIR  = RUNS_DIR / "plummer" / "phi"
JOINT_RUN_DIR = RUNS_DIR / "plummer" / "joint"

result = run_joint_finetuning(
    config=cfg,
    data_path=DATA_PATH,
    df_run_dir=DF_RUN_DIR,
    phi_run_dir=PHI_RUN_DIR,
    run_dir=JOINT_RUN_DIR,
)

print(f"\nJoint fine-tuning complete. Final step: {result['final_step']}")

## 3. Inspect Joint Training Metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

metrics_path = JOINT_RUN_DIR / "metrics.csv"

with metrics_path.open() as f:
    reader = csv.DictReader(f)
    rows = [r for r in reader]

steps = np.array([float(r["step"]) for r in rows])
loss  = np.array([float(r["loss"]) for r in rows])
nll   = np.array([float(r["nll"]) for r in rows])
cbe   = np.array([float(r["cbe"]) for r in rows])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, loss, lw=1.2)
axes[0].set_xlabel("step"); axes[0].set_ylabel("total loss")
axes[0].set_title("Joint Loss"); axes[0].grid(True, alpha=0.2)

axes[1].plot(steps, nll, lw=1.2, color="tab:blue")
axes[1].set_xlabel("step"); axes[1].set_ylabel("NLL")
axes[1].set_title("NLL Component"); axes[1].grid(True, alpha=0.2)

axes[2].plot(steps, cbe, lw=1.2, color="tab:red")
axes[2].set_xlabel("step"); axes[2].set_ylabel("CBE loss")
axes[2].set_title("CBE Component"); axes[2].grid(True, alpha=0.2)

fig.tight_layout()
plt.show()

## 4. Evaluate with Joint-Tuned Model

After joint fine-tuning, the DF and Phi sub-directories under `runs/plummer/joint/` are compatible with the standard eval scripts.

In [ ]:
from experiments.eval_phi import run_eval_phi

eval_result = run_eval_phi(
    data_path=DATA_PATH,
    df_run_dir=JOINT_RUN_DIR / "df",
    phi_run_dir=JOINT_RUN_DIR / "phi",
)

print("\nEvaluation stats:")
for k, v in eval_result["stats"].items():
    print(f"  {k}: {v}")